# Compare your own run with the published curves

The same comparison as the website's *Check your own run*, from Python. It tells you which published curves are fair to compare with (their definitions agree with yours) and how far your curve is from them. Nothing is interpolated beyond the range of either curve.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import sim_highline as sh
df = sh.load()

## 1. Your curve

`read_curve` accepts comma, semicolon (with decimal commas), tab or space separated rows, a header, comment lines and an optional error column. Here we use a small file shipped with the repository; replace the path with yours. Values are log10 as plotted (log stellar mass in Msun and log number density per cMpc³ per dex for the stellar mass function).

In [ ]:
x, y, err = sh.read_curve("tests/user-curves/3-bom-crlf-header.csv")
print(len(x), "points;", "x from", x.min(), "to", x.max())

## 2. Say what your numbers mean

A definition you leave out can never be matched, so with none declared no curve is called comparable. Use the keys of the dataset's definition columns (`imf`, `massDefinition`, `densityFrame`, `population`, …); the site's *Your run* panel lists the choices with a plain explanation of each.

In [ ]:
mine = {"imf": "chabrier03", "densityFrame": "comoving", "massDefinition": "aperture-50pkpc"}

## 3. Compare

In [ ]:
res = sh.compare_run(x, y, "gsmf", z=0.1, definitions=mine, sigma=err, df=df)
print(sh.describe(res, "gsmf", 0.1))
res[["source", "run", "tier", "verdict", "median_dy", "rms_dy", "n_bins"]].head(10)

## 4. Look at it

Your run against the curves that are fair to compare with. Offsets are only verdicts where the definitions agree.

In [ ]:
fair = res[res.verdict != "not-comparable"].head(6)
fig, ax = plt.subplots(figsize=(6, 4))
for rid in fair.record_id:
    xs, ys, *_ = sh.curve(df, rid)
    ax.plot(xs, ys, lw=1, alpha=.7)
ax.errorbar(x, y, yerr=None if np.isnan(err).all() else err, color="k", lw=2, label="your run")
ax.set_xlabel("log stellar mass"); ax.set_ylabel("log number density"); ax.legend(); plt.show()

## 5. Cite what you used

`sh.bibtex(selection)` writes BibTeX for exactly the curves you plotted. Cite the original papers as well as the dataset.

In [ ]:
print(sh.bibtex(df[df.record_id.isin(fair.record_id)])[:600])